# Modelo 2 — XGBoost Vanilla: predicción de partidos internacionales

Segundo modelo del proyecto: un **XGBoost "vanilla"** (parámetros prácticamente por
defecto, sin tuning) que sirve como baseline de Machine Learning clásico para comparar
contra el **Modelo 1 (Bayesiano MCMC en PyMC)**.

**Este notebook reutiliza EXACTAMENTE:**
- El mismo dataset: [martj42/international_results](https://github.com/martj42/international_results)
- El mismo target: `0` = gana local, `1` = empate, `2` = gana visitante
- El mismo feature engineering del Modelo 1 (~180 variables, sin data leakage)
- El mismo split temporal 80% / 20%

No se rediseña nada del feature engineering ni se elimina ninguna variable — el único
cambio real es el modelo: de regresión logística Bayesiana con MCMC a `XGBClassifier`
sin optimización de hiperparámetros.


## 1. Imports

**Qué hace:** carga las librerías necesarias para todo el pipeline.

**Por qué:** `xgboost` aporta el modelo, `joblib` es el estándar para serializar
modelos de scikit-learn/XGBoost (más eficiente que `pickle` para arrays de numpy
grandes), y `seaborn` se usa para una matriz de confusión más clara.

**Ventaja:** mismas librerías base que el Modelo 1 (pandas/numpy/matplotlib/sklearn)
más las específicas de este modelo, para que ambos notebooks sean fáciles de comparar
lado a lado.

In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================
!pip install -q xgboost joblib

import warnings
warnings.filterwarnings("ignore")

import os
import subprocess
import time
import pickle
import difflib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, log_loss, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
)

import xgboost as xgb
from xgboost import XGBClassifier
import joblib

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("xgboost version:", xgb.__version__)
print("pandas version:", pd.__version__)


## 2. Carga del dataset

**Qué hace:** clona (o reutiliza) el mismo repositorio de GitHub usado en el Modelo 1
y busca de forma flexible el CSV de resultados históricos.

**Por qué:** para que las comparaciones entre modelos sean justas, ambos deben partir
exactamente del mismo dataset crudo — código idéntico al del Modelo 1, sin modificar.

**Ventaja:** si el repo cambia de nombre de archivo o de estructura de carpetas,
`find_results_csv` lo sigue encontrando por sus columnas, sin romper el notebook.

In [ ]:
# ============================================================
# 2. CLONAR REPOSITORIO Y CARGAR EL DATASET (identico al Modelo 1)
# ============================================================
# Pega aqui el link del repo de GitHub que contiene los datos historicos.
REPO_URL = "https://github.com/martj42/international_results"

REPO_DIR = "/content/international_results_repo"

if not os.path.exists(REPO_DIR):
    try:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    except Exception as e:
        print("No se pudo clonar el repo automaticamente:", e)
else:
    print("El repositorio ya existe localmente, se omite el clonado.")


def find_results_csv(search_dirs):
    """
    Busca de forma flexible un archivo CSV de resultados dentro de una lista de
    directorios. Prioriza nombres tipicos ('results.csv') y si no los encuentra,
    busca cualquier CSV que contenga las columnas esperadas (home_team, away_team, ...).
    """
    candidate_names = ["results.csv", "result.csv", "matches.csv", "international_results.csv"]
    csv_files = []
    for base_dir in search_dirs:
        if not os.path.isdir(base_dir):
            continue
        for root, _, files in os.walk(base_dir):
            for f in files:
                if f.lower().endswith(".csv"):
                    csv_files.append(os.path.join(root, f))

    # 1. Buscar coincidencia exacta por nombre
    for name in candidate_names:
        for path in csv_files:
            if os.path.basename(path).lower() == name:
                return path

    # 2. Si no hay coincidencia por nombre, buscar por columnas esperadas
    expected_cols = {"home_team", "away_team", "home_score", "away_score"}
    for path in csv_files:
        try:
            sample = pd.read_csv(path, nrows=5)
            if expected_cols.issubset(set(c.lower() for c in sample.columns)):
                return path
        except Exception:
            continue

    raise FileNotFoundError(
        "No se encontro un CSV con las columnas esperadas. "
        "Revisa REPO_URL o coloca manualmente la ruta del archivo en csv_path."
    )


# Busca primero en el repo clonado y, como respaldo, en el directorio actual
csv_path = find_results_csv([REPO_DIR, "."])
print("Archivo de resultados encontrado en:", csv_path)

raw_df = pd.read_csv(csv_path)
raw_df.columns = [c.strip().lower() for c in raw_df.columns]
print("Shape original:", raw_df.shape)
raw_df.head()


## 3. Limpieza + Feature Engineering (idéntico al Modelo 1 — sin cambios)

**Qué hace:** limpia el dataset, crea el `target`, y genera las ~180 variables de
forma (rolling de 3/5/10/15/20 partidos, para local/visitante/diferencia) exactamente
igual que en el Modelo 1.

**Por qué:** el objetivo de este notebook es comparar **el mismo conjunto de variables**
bajo dos modelos distintos (Bayesiano vs. XGBoost). Cualquier diferencia en resultados
debe venir del modelo, no del feature engineering — por eso este bloque es una copia
literal del Modelo 1, sin tocar una sola línea de la lógica de construcción de
variables.

**Ventaja:** evita data leakage (cada variable de forma usa `shift(1)` antes del
`rolling`, así que el partido actual nunca influye en sus propias variables) y deja el
dataset listo con `all_feature_cols` (~180 columnas) para el split y el modelo.

In [ ]:
# ============================================================
# 3.1 LIMPIEZA DEL DATASET (identico al Modelo 1)
# ============================================================
df = raw_df.copy()

# Normalizar nombres de columnas por si vienen con variantes
rename_map = {
    "hometeam": "home_team", "awayteam": "away_team",
    "homescore": "home_score", "awayscore": "away_score",
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

required_cols = ["date", "home_team", "away_team", "home_score", "away_score"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Faltan columnas obligatorias en el dataset: {missing}")

# Convertir fecha y descartar filas sin fecha valida
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"])

# Ordenar cronologicamente (necesario para que el feature engineering no tenga leakage)
df = df.sort_values("date").reset_index(drop=True)

# Eliminar partidos sin marcador valido
df = df.dropna(subset=["home_score", "away_score"])
df["home_score"] = df["home_score"].astype(int)
df["away_score"] = df["away_score"].astype(int)

# Columnas opcionales manejadas de forma flexible
if "tournament" not in df.columns:
    df["tournament"] = "Unknown"

if "neutral" not in df.columns:
    df["neutral"] = False
else:
    df["neutral"] = (
        df["neutral"].astype(str).str.upper().map({"TRUE": True, "FALSE": False}).fillna(False)
    )
df["neutral"] = df["neutral"].astype(int)

# id unico de partido (se usa para reconstruir las features mas adelante)
df["match_id"] = np.arange(len(df))

# Variable objetivo (target)
# 0 = gana home_team | 1 = empate | 2 = gana away_team
conditions = [
    df["home_score"] > df["away_score"],
    df["home_score"] == df["away_score"],
    df["home_score"] < df["away_score"],
]
df["target"] = np.select(conditions, [0, 1, 2]).astype(int)

print(f"Dataset limpio: {df.shape[0]} partidos, {df.shape[1]} columnas")
print(df["target"].value_counts(normalize=True).rename("proporcion"))
df[["date", "home_team", "away_team", "home_score", "away_score", "target"]].head()


In [ ]:
# ============================================================
# 3.2 FEATURE ENGINEERING (SIN DATA LEAKAGE) - identico al Modelo 1
# ============================================================
WINDOWS = [3, 5, 10, 15, 20]

# --- Formato largo: una fila por equipo y partido ---
long_cols = ["match_id", "date", "home_team", "away_team", "home_score", "away_score", "tournament", "neutral"]

home_long = df[long_cols].rename(columns={
    "home_team": "team", "away_team": "opponent",
    "home_score": "goals_for", "away_score": "goals_against",
})
home_long["is_home"] = 1

away_long = df[long_cols].rename(columns={
    "away_team": "team", "home_team": "opponent",
    "away_score": "goals_for", "home_score": "goals_against",
})
away_long["is_home"] = 0

long_df = pd.concat([home_long, away_long], ignore_index=True)
long_df = long_df.sort_values(["team", "date", "match_id"]).reset_index(drop=True)

# --- Metricas base por partido (antes de aplicar ventanas) ---
long_df["goal_diff"] = long_df["goals_for"] - long_df["goals_against"]
long_df["win"] = (long_df["goals_for"] > long_df["goals_against"]).astype(int)
long_df["draw"] = (long_df["goals_for"] == long_df["goals_against"]).astype(int)
long_df["loss"] = (long_df["goals_for"] < long_df["goals_against"]).astype(int)
long_df["points"] = long_df["win"] * 3 + long_df["draw"] * 1
long_df["clean_sheet"] = (long_df["goals_against"] == 0).astype(int)
long_df["failed_to_score"] = (long_df["goals_for"] == 0).astype(int)

# 9 metricas "mean/rate" + 3 metricas de desviacion estandar = 12 metricas por ventana
BASE_METRICS = ["goals_for", "goals_against", "goal_diff", "points",
                "win", "draw", "loss", "clean_sheet", "failed_to_score"]
STD_METRICS = ["goals_for", "goals_against", "goal_diff"]

# --- Rolling con shift(1): usa SOLO partidos anteriores al actual ---
feature_cols_long = []
grouped = long_df.groupby("team", group_keys=False)

t0 = time.time()
new_cols = {}
for w in WINDOWS:
    for col in BASE_METRICS:
        feat_name = f"{col}_mean_{w}"
        new_cols[feat_name] = grouped[col].apply(lambda s: s.shift(1).rolling(w, min_periods=1).mean())
        feature_cols_long.append(feat_name)
    for col in STD_METRICS:
        feat_name = f"{col}_std_{w}"
        new_cols[feat_name] = grouped[col].apply(lambda s: s.shift(1).rolling(w, min_periods=1).std())
        feature_cols_long.append(feat_name)

long_df = pd.concat([long_df, pd.DataFrame(new_cols, index=long_df.index)], axis=1)
print(f"Variables de forma por equipo calculadas en {time.time() - t0:.1f}s -> {len(feature_cols_long)} columnas")

# --- Volver a formato ancho: features del local, del visitante y diferencias ---
rename_home = {c: f"home_{c}" for c in feature_cols_long}
rename_away = {c: f"away_{c}" for c in feature_cols_long}

home_features = long_df.loc[long_df.is_home == 1, ["match_id"] + feature_cols_long].rename(columns=rename_home)
away_features = long_df.loc[long_df.is_home == 0, ["match_id"] + feature_cols_long].rename(columns=rename_away)

df = df.merge(home_features, on="match_id", how="left").merge(away_features, on="match_id", how="left")

diff_data = {f"diff_{c}": df[f"home_{c}"] - df[f"away_{c}"] for c in feature_cols_long}
df = pd.concat([df, pd.DataFrame(diff_data, index=df.index)], axis=1)
diff_cols = list(diff_data.keys())

all_feature_cols = [f"home_{c}" for c in feature_cols_long] + [f"away_{c}" for c in feature_cols_long] + diff_cols
print(f"Total de variables generadas: {len(all_feature_cols)}")

# Partidos al inicio del historico de un equipo no tienen ventana previa -> NaN.
# Se rellenan con 0 (equivalente a "sin informacion previa disponible").
df[all_feature_cols] = df[all_feature_cols].fillna(0)

assert len(all_feature_cols) > 100, "Se esperaban mas de 100 variables"
print(f"\nUsando TODAS las {len(all_feature_cols)} variables generadas (sin recortar ninguna).")
df[all_feature_cols].describe().T.head()


## 4. Train / Test Split (temporal, idéntico al Modelo 1)

**Qué hace:** separa los partidos usando el primer 80% cronológico para entrenar y el
último 20% para evaluar.

**Por qué:** un split aleatorio (`train_test_split`) mezclaría partidos futuros dentro
del entrenamiento y partidos antiguos en el test, lo cual es un data leakage temporal
sutil pero real (el modelo "vería el futuro"). El split cronológico simula el
escenario real: predecir partidos que aún no han ocurrido usando solo información del
pasado.

**Ventaja:** los resultados de este modelo son directamente comparables con los del
Modelo 1, porque ambos usan exactamente los mismos partidos de train y de test.

In [ ]:
# ============================================================
# 4. SEPARACION TEMPORAL TRAIN / TEST (80% / 20%)
# ============================================================
df_model = df.sort_values("date").reset_index(drop=True)

split_idx = int(len(df_model) * 0.8)

train_df = df_model.iloc[:split_idx].copy()
test_df = df_model.iloc[split_idx:].copy()

print(f"Train: {train_df.shape[0]} partidos "
      f"({train_df['date'].min().date()} a {train_df['date'].max().date()})")
print(f"Test:  {test_df.shape[0]} partidos "
      f"({test_df['date'].min().date()} a {test_df['date'].max().date()})")

y_train = train_df["target"].values.astype(int)
y_test = test_df["target"].values.astype(int)


## 5. Escalado de variables — **no es necesario para XGBoost**

**Qué hace este bloque:** en vez de escalar, simplemente convierte `all_feature_cols`
a arrays de numpy para alimentar al modelo.

**Por qué no se necesita `StandardScaler` aquí (a diferencia del Modelo 1):**

- XGBoost es un **modelo basado en árboles de decisión**. Cada árbol decide sus
  splits preguntando cosas como *"¿`diff_goal_diff_mean_10` > 0.35?"*. Ese umbral se
  ajusta a la escala real de la variable; **el orden relativo de los valores es lo
  único que importa**, no su magnitud absoluta.
- Escalar una variable (restar la media, dividir por la desviación estándar) es una
  **transformación monótona** — no cambia el orden de los valores, así que **no
  cambia ni un solo split del árbol ni la importancia de las variables**. El modelo
  entrenado sería matemáticamente equivalente con o sin escalado.
- El escalado sí importa en el Modelo 1 (regresión logística Bayesiana) porque ahí los
  coeficientes (`beta`) se combinan linealmente y su prior (`Normal(0, sigma)`) asume
  que todas las variables están en una escala comparable. XGBoost no tiene ese
  supuesto: no hay coeficientes lineales ni priors sobre magnitudes.
- Evitar el escalado también simplifica el pipeline de predicción futura: no hace
  falta guardar ni aplicar un `scaler` en `predict_future_match_xgboost`.

**Conclusión:** se usa `all_feature_cols` tal cual, sin ninguna transformación de
escala. El diccionario de artifacts guardará `scaler_xgb = None` para dejar explícito
que este modelo no lo necesita (y para que la función de carga sepa que no hay que
aplicar ninguno).

In [ ]:
# ============================================================
# 5. "ESCALADO" - NO NECESARIO PARA XGBOOST (ver explicacion arriba)
# ============================================================
X_train = train_df[all_feature_cols].values.astype(float)
X_test = test_df[all_feature_cols].values.astype(float)

# No se ajusta ningun StandardScaler: se deja explicito como None para el pipeline
# de guardado/carga (Secciones 12-13), documentando por que no aplica aqui.
scaler_xgb = None

print("Shape para XGBoost -> train:", X_train.shape, "| test:", X_test.shape)
print(f"Variables usadas: {len(all_feature_cols)} (todas, sin escalar, sin recortar)")


## 6. Modelo XGBoost Vanilla

**Qué hace:** define un `XGBClassifier` con hiperparámetros prácticamente por defecto
(sin `GridSearchCV`, sin `RandomizedSearchCV`, sin búsqueda de hiperparámetros de
ningún tipo).

**Por qué "vanilla":** este modelo funciona como **baseline** — el punto de partida
honesto contra el cual medir tanto el Modelo 1 (Bayesiano) como futuras versiones
optimizadas de XGBoost. Si se afinara aquí, dejaría de ser un baseline comparable.

**Ventaja:** un vanilla model es rápido de entrenar, fácil de razonar, y establece una
cota de referencia clara: cualquier modelo más complejo debería superar a este para
justificar su complejidad adicional.

In [ ]:
# ============================================================
# 6. MODELO XGBOOST VANILLA (sin tuning de hiperparametros)
# ============================================================
xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

print(xgb_model)


## 7. Entrenamiento

**Qué hace:** entrena el modelo con `X_train`/`y_train`, usando **todas** las ~180
variables generadas.

**Por qué:** a diferencia del Modelo 1 (donde se usó un prior jerárquico de
regularización para poder manejar 180 variables correlacionadas dentro de un MCMC),
XGBoost maneja de forma nativa cientos de variables correlacionadas: en cada split
solo considera un subconjunto de variables (`colsample_bytree`, aunque aquí se deja en
su valor por defecto), por lo que la colinealidad no es un problema estructural como
en un modelo lineal.

**Ventaja:** entrenamiento típicamente en segundos (mucho más rápido que el MCMC del
Modelo 1), sin necesidad de reducir variables ni ajustar priors.

In [ ]:
# ============================================================
# 7. ENTRENAMIENTO (usando TODAS las variables)
# ============================================================
t0 = time.time()
xgb_model.fit(X_train, y_train)
print(f"Entrenamiento completado en {time.time() - t0:.1f}s")


## 8. Evaluación en el conjunto de test

**Qué hace:** calcula accuracy, log loss, precision/recall/F1 (macro y weighted),
el `classification_report` completo, y la matriz de confusión.

**Por qué cada métrica:**
- **Accuracy:** proporción de partidos acertados; fácil de interpretar pero engañosa
  si las clases están desbalanceadas (aquí "empate" es la clase minoritaria).
- **Log loss:** penaliza probabilidades mal calibradas, no solo la clase predicha —
  importante porque el objetivo final es obtener probabilidades útiles (ver Sección 10).
- **Precision / Recall / F1 (macro y weighted):** el *macro* promedia igual las 3
  clases (bueno para ver qué tan bien predice "empate", la clase difícil); el
  *weighted* pondera por soporte (refleja mejor el desempeño global).
- **Matriz de confusión:** muestra en qué se equivoca el modelo (p.ej. si confunde
  empates con victorias del local, un patrón común en fútbol).

In [ ]:
# ============================================================
# 8. EVALUACION EN EL CONJUNTO DE TEST
# ============================================================
y_pred = xgb_model.predict(X_test)
y_proba = xgb_model.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
ll = log_loss(y_test, y_proba, labels=[0, 1, 2])

precision_macro = precision_score(y_test, y_pred, average="macro", zero_division=0)
recall_macro = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)

precision_weighted = precision_score(y_test, y_pred, average="weighted", zero_division=0)
recall_weighted = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1_weighted = f1_score(y_test, y_pred, average="weighted", zero_division=0)

print(f"Accuracy:  {acc:.4f}")
print(f"Log Loss:  {ll:.4f}")
print()
print(f"Precision (macro):    {precision_macro:.4f}   | (weighted): {precision_weighted:.4f}")
print(f"Recall    (macro):    {recall_macro:.4f}   | (weighted): {recall_weighted:.4f}")
print(f"F1-score  (macro):    {f1_macro:.4f}   | (weighted): {f1_weighted:.4f}")

print("\nClassification report:")
print(classification_report(
    y_test, y_pred,
    target_names=["Gana local", "Empate", "Gana visitante"],
    zero_division=0,
))

cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=["Real: local", "Real: empate", "Real: visitante"],
    columns=["Pred: local", "Pred: empate", "Pred: visitante"],
)
print("Matriz de confusion:")
print(cm_df)


In [ ]:
# Matriz de confusion (grafica con seaborn)
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Local", "Empate", "Visitante"],
            yticklabels=["Local", "Empate", "Visitante"])
plt.xlabel("Predicho")
plt.ylabel("Real")
plt.title("Matriz de confusion - XGBoost Vanilla")
plt.tight_layout()
plt.show()


## 9. Importancia de variables (Feature Importance)

**Qué hace:** extrae `feature_importances_` del modelo entrenado y grafica las 20
variables con mayor importancia.

**Por qué:** a diferencia del Modelo 1 (donde había que mirar los coeficientes
`beta` uno por uno), XGBoost calcula la importancia de cada variable de forma nativa
como parte del entrenamiento (basada en la ganancia media que aporta cada variable
cuando se usa para un split).

**Ventaja:** permite explicar el modelo de forma directa en un portafolio o video —
por ejemplo, confirmar que las métricas de forma reciente (ventanas de 10-20 partidos)
dominan sobre las de muy corto plazo (3 partidos), lo cual es consistente con la
intuición futbolística.

In [ ]:
# ============================================================
# 9. FEATURE IMPORTANCE - TOP 20 VARIABLES
# ============================================================
importances = pd.Series(xgb_model.feature_importances_, index=all_feature_cols)
importances = importances.sort_values(ascending=False)

top20 = importances.head(20)
print("Top 20 variables mas importantes:")
print(top20)

plt.figure(figsize=(8, 8))
plt.barh(top20.index[::-1], top20.values[::-1], color="#2ca02c")
plt.xlabel("Importancia (gain)")
plt.title("Top 20 variables mas importantes - XGBoost Vanilla")
plt.tight_layout()
plt.show()


## 10. Predicción de partidos futuros: `predict_future_match_xgboost`

**Qué hace:** `create_match_features` construye el mismo esquema de ~180 variables
(`home_*`, `away_*`, `diff_*`) para un partido hipotético, usando la forma más
reciente de cada equipo (últimos partidos disponibles en `long_df`, incluyendo el más
reciente ya que se proyecta un partido que aún no ocurrió). `predict_future_match_xgboost`
valida los nombres de equipo, construye las features, y usa el modelo ya entrenado
para predecir.

**Por qué la validación de nombres:** los nombres deben coincidir exactamente con los
usados en el dataset (en inglés). Si escribes "Francia" en vez de "France", es mejor
detenerse con un mensaje claro y sugerencias que devolver una predicción silenciosa
sin sentido.

**Ventaja:** la misma función sirve para cualquier par de equipos del dataset (ver
Sección 14 con 4 ejemplos), y es la pieza reutilizable clave para producción o para
un futuro dashboard/API de predicciones.

In [ ]:
# ============================================================
# 10. CREATE_MATCH_FEATURES + PREDICT_FUTURE_MATCH_XGBOOST
# ============================================================
# NOTA DE DISENO: estas funciones NO capturan long_df/WINDOWS/all_feature_cols como
# valores por defecto (que Python fijaria en el momento de DEFINIR la funcion).
# En su lugar los leen como variables globales dentro del cuerpo de la funcion, es
# decir, en el momento en que se LLAMAN. Esto permite definir las funciones aqui
# (Seccion 10) y que sigan funcionando mas adelante sin importar si long_df fue
# creado por el entrenamiento (Seccion 3) o por load_xgboost_model() (Seccion 13).


def validate_team_name(team_name):
    """
    Verifica que team_name exista tal cual en el dataset (usa el long_df global
    vigente). Si no existe, sugiere nombres parecidos (ej. 'Francia' -> 'France')
    y lanza un error claro en vez de seguir con datos vacios/incorrectos.
    """
    known_teams = sorted(set(long_df["team"].unique()))
    if team_name in known_teams:
        return team_name

    suggestions = difflib.get_close_matches(team_name, known_teams, n=5, cutoff=0.6)
    msg = f"El equipo '{team_name}' no existe en el dataset."
    if suggestions:
        msg += f" Quisiste decir: {suggestions}?"
    else:
        msg += " No se encontraron nombres parecidos; revisa la lista de equipos del dataset."
    raise ValueError(msg)


def get_latest_team_form(team_name, long_df):
    """
    Calcula las metricas de forma de un equipo usando sus ULTIMOS partidos
    disponibles en el historico (incluye el partido mas reciente, ya que se
    esta proyectando un partido futuro que todavia no se jugo). Usa el WINDOWS
    global vigente (definido en el feature engineering o restaurado al cargar).
    """
    hist = long_df[long_df["team"] == team_name].sort_values("date")
    stats = {}
    for w in WINDOWS:
        recent = hist.tail(w)
        for col in BASE_METRICS:
            stats[f"{col}_mean_{w}"] = recent[col].mean() if len(recent) > 0 else 0.0
        for col in STD_METRICS:
            stats[f"{col}_std_{w}"] = recent[col].std() if len(recent) > 1 else 0.0
    return stats


def create_match_features(home_team, away_team, neutral=1):
    """
    Construye el vector de variables (las ~180 completas, en el mismo orden que
    all_feature_cols) para un partido hipotetico entre home_team y away_team.
    """
    home_stats = get_latest_team_form(home_team, long_df)
    away_stats = get_latest_team_form(away_team, long_df)

    row = {}
    for col, val in home_stats.items():
        row[f"home_{col}"] = val
    for col, val in away_stats.items():
        row[f"away_{col}"] = val
    for col in home_stats:
        row[f"diff_{col}"] = home_stats[col] - away_stats[col]
    # 'neutral' se deja disponible por si se agrega como feature en una version futura
    row["_neutral"] = neutral

    match_df = pd.DataFrame([row])
    for c in all_feature_cols:
        if c not in match_df.columns:
            match_df[c] = 0.0
    match_df = match_df[all_feature_cols].fillna(0.0)

    return match_df[all_feature_cols].values.astype(float)


CLASS_NAMES = ["Gana home_team", "Empate", "Gana away_team"]


def predict_future_match_xgboost(home_team, away_team, neutral=1, model=None,
                                  export_csv=True, plot=True):
    """
    Predice el resultado de un partido futuro home_team vs away_team con el
    modelo XGBoost ya entrenado/cargado. Devuelve un dict con las probabilidades
    y la clase predicha, muestra una tabla clara, una grafica de barras, y
    exporta un CSV con nombre automatico.
    """
    model = model if model is not None else xgb_model

    home_team = validate_team_name(home_team)
    away_team = validate_team_name(away_team)

    X_match = create_match_features(home_team, away_team, neutral=neutral)
    proba = model.predict_proba(X_match)[0]
    pred_class = int(np.argmax(proba))

    table = pd.DataFrame({
        "resultado": [f"{home_team} gana", "Empate", f"{away_team} gana"],
        "probabilidad": proba,
    })
    table["probabilidad_%"] = (table["probabilidad"] * 100).round(2)

    print(f"\nPrediccion XGBoost Vanilla: {home_team} vs {away_team}\n")
    print(table.to_string(index=False))
    print(f"\nClase predicha: {pred_class} ({CLASS_NAMES[pred_class]})")

    if plot:
        plt.figure(figsize=(6, 4))
        colors = ["#1f77b4", "#7f7f7f", "#d62728"]
        plt.bar(table["resultado"], table["probabilidad_%"], color=colors)
        plt.ylabel("Probabilidad (%)")
        plt.title(f"{home_team} vs {away_team} (XGBoost Vanilla)")
        for i, v in enumerate(table["probabilidad_%"]):
            plt.text(i, v + 1, f"{v:.1f}%", ha="center", fontweight="bold")
        plt.ylim(0, max(table["probabilidad_%"]) + 15)
        plt.tight_layout()
        plt.show()

    if export_csv:
        safe_home = home_team.replace(" ", "_")
        safe_away = away_team.replace(" ", "_")
        csv_name = f"prediction_{safe_home}_vs_{safe_away}_xgboost.csv"
        table[["resultado", "probabilidad"]].to_csv(csv_name, index=False)
        print(f"\nArchivo exportado: {csv_name}")

    return {
        "home_team": home_team,
        "away_team": away_team,
        "proba_home": float(proba[0]),
        "proba_draw": float(proba[1]),
        "proba_away": float(proba[2]),
        "predicted_class": pred_class,
        "predicted_label": CLASS_NAMES[pred_class],
        "table": table,
    }


## 11. Visualización de probabilidades

**Qué hace:** ya está integrada dentro de `predict_future_match_xgboost` (parámetro
`plot=True`): genera una gráfica de barras con las 3 probabilidades por cada llamada.

**Por qué está dentro de la función y no separada:** mantiene la función
autocontenida y reutilizable — cualquiera que la importe obtiene tabla + gráfica +
CSV en una sola llamada, sin tener que acordarse de graficar por separado.

**Ventaja:** consistencia visual entre todas las predicciones (mismos colores, mismo
formato), útil para un video o reporte donde se muestran varios partidos seguidos.

## 12. Guardar el modelo (local, no en Google Drive)

**Qué hace:** guarda el modelo entrenado con `joblib` (formato estándar y eficiente
para modelos de scikit-learn/XGBoost) y un diccionario de artifacts con todo lo
necesario para reutilizarlo después, sin tener que reentrenar ni rehacer el feature
engineering.

**Por qué `joblib` para el modelo y `pickle` para los artifacts:** `joblib` está
optimizado para objetos con arrays de numpy grandes (como los árboles internos de
XGBoost), mientras que `artifacts_xgboost.pkl` guarda estructuras más heterogéneas
(listas, un DataFrame, `None`) donde `pickle` estándar es suficiente y más simple.

**Ventaja:** cualquiera puede cargar `modelo_xgboost_vanilla.joblib` +
`artifacts_xgboost.pkl` en una sesión nueva de Colab (o local) y predecir de inmediato,
sin necesidad de volver a descargar el dataset ni reentrenar.

In [ ]:
# ============================================================
# 12. GUARDAR MODELO Y ARTIFACTS
# ============================================================
MODEL_PATH = "modelo_xgboost_vanilla.joblib"
ARTIFACTS_PATH = "artifacts_xgboost.pkl"

joblib.dump(xgb_model, MODEL_PATH)
print("Modelo guardado en:", MODEL_PATH)

artifacts_xgb = {
    "scaler": scaler_xgb,               # None: XGBoost no necesita escalado (ver Seccion 5)
    "feature_cols": all_feature_cols,   # las ~180 variables usadas por el modelo (todas)
    "team_history": long_df,            # historial largo por equipo, necesario para partidos futuros
    "class_names": CLASS_NAMES,
    "windows": WINDOWS,
    "base_metrics": BASE_METRICS,
    "std_metrics": STD_METRICS,
    "known_teams": sorted(set(long_df["team"].unique())),
}

with open(ARTIFACTS_PATH, "wb") as f:
    pickle.dump(artifacts_xgb, f)

print("Artifacts guardados en:", ARTIFACTS_PATH)


## 13. Cargar el modelo guardado (sin reentrenar)

**Qué hace:** `load_xgboost_model()` carga el `.joblib` y el `.pkl`, y restaura las
variables globales que necesita `predict_future_match_xgboost` (`all_feature_cols`,
`long_df`, `WINDOWS`, etc.) para poder predecir en una sesión nueva sin repetir
ninguna de las Secciones 2 a 9.

**Por qué:** separar entrenamiento de inferencia es una práctica estándar en
proyectos de ML — el entrenamiento es costoso y se hace una vez; la inferencia debe
ser rápida y repetible.

**Ventaja:** en una demo o video, puedes abrir un notebook nuevo, correr solo los
imports (Sección 1) y esta celda, y predecir partidos en segundos.

In [ ]:
# ============================================================
# 13. CARGAR MODELO GUARDADO (para usar en una sesion nueva sin reentrenar)
# ============================================================
def load_xgboost_model(model_path="modelo_xgboost_vanilla.joblib",
                        artifacts_path="artifacts_xgboost.pkl", set_globals=True):
    """
    Carga el modelo XGBoost y sus artifacts desde disco. Si set_globals=True
    (por defecto), tambien actualiza las variables globales que usa
    predict_future_match_xgboost (all_feature_cols, long_df, WINDOWS, etc.),
    para poder predecir sin haber corrido el entrenamiento en esta sesion.
    """
    model = joblib.load(model_path)
    with open(artifacts_path, "rb") as f:
        artifacts = pickle.load(f)

    if set_globals:
        global all_feature_cols, long_df, WINDOWS, BASE_METRICS, STD_METRICS
        global CLASS_NAMES, scaler_xgb

        all_feature_cols = artifacts["feature_cols"]
        long_df = artifacts["team_history"]
        WINDOWS = artifacts["windows"]
        BASE_METRICS = artifacts["base_metrics"]
        STD_METRICS = artifacts["std_metrics"]
        CLASS_NAMES = artifacts["class_names"]
        scaler_xgb = artifacts["scaler"]

    print(f"Modelo cargado desde: {model_path}")
    print(f"Artifacts cargados desde: {artifacts_path} ({len(artifacts['feature_cols'])} variables)")
    return model, artifacts


# Ejemplo de uso en una sesion nueva (descomenta para usarlo en lugar de reentrenar):
# xgb_model, artifacts_xgb = load_xgboost_model()


## 14. Ejemplos de predicción

Cuatro partidos de ejemplo, incluyendo el objetivo del proyecto (Francia vs Paraguay).

In [ ]:
# Ejemplo 1: partido objetivo del proyecto
result_france_paraguay = predict_future_match_xgboost("France", "Paraguay", neutral=1)


In [ ]:
# Ejemplo 2
result_mexico_england = predict_future_match_xgboost("Mexico", "England", neutral=1)


In [ ]:
# Ejemplo 3
result_brazil_norway = predict_future_match_xgboost("Brazil", "Norway", neutral=1)


In [ ]:
# Ejemplo 4
result_portugal_spain = predict_future_match_xgboost("Portugal", "Spain", neutral=1)


## Resumen: Modelo 1 (Bayesiano MCMC) vs. Modelo 2 (XGBoost Vanilla)

| Aspecto | Modelo 1 — Bayesiano MCMC | Modelo 2 — XGBoost Vanilla |
|---|---|---|
| Dataset | martj42/international_results | **Idéntico** |
| Target | 0/1/2 (local/empate/visitante) | **Idéntico** |
| Feature engineering | ~180 variables, rolling 3/5/10/15/20, sin leakage | **Idéntico, sin cambios** |
| Split | Temporal 80/20 | **Idéntico** |
| Escalado | `StandardScaler` (necesario: modelo lineal) | No aplica (árboles son invariantes a la escala) |
| Selección de variables | Todas (con prior jerárquico de regularización) | Todas (XGBoost maneja colinealidad de forma nativa) |
| Entrenamiento | MCMC/NUTS (minutos a horas) | Boosting de árboles (segundos) |
| Salida | Probabilidades posteriores + incertidumbre (HDI) | Probabilidades puntuales (`predict_proba`) |
| Interpretabilidad | Coeficientes `beta` por variable | `feature_importances_` nativo |

Ambos modelos comparten dataset, target, feature engineering y split — la única
variable que cambia es el algoritmo de modelado, lo cual permite una comparación
justa de accuracy, log loss y calibración de probabilidades entre un enfoque
Bayesiano y uno de *gradient boosting* clásico.
